# Newton and secant methods

In [ ]:
#    APM41012EP course notebook - Chapter 6 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Newton and secant methods
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "seaborn"

## Iteration of Newton's method

We propose to test Newton's method on the function:

$$f(x) = x^3 -2 x -5 $$

In [ ]:
def f(x):
    return x**3 - 2*x - 5

def df(x):
    return 3*x**2 - 2

In [ ]:
def newton(f, df, x0, xstar, tol=1.e-12, nitmax=30):

    # initialisation
    x = np.zeros(nitmax+1)
    err = np.zeros(nitmax+1)
    err_res = np.zeros(nitmax+1)
    diff_x = np.zeros(nitmax)
    x[0] = x0
    err[0] = abs(x0 - xstar)
    err_res[0] = abs(f(x0))

    # Newton iteration
    for i in range(1, nitmax+1):
        x[i] = x[i-1] - f(x[i-1])/df(x[i-1])
        err[i] = abs(x[i]-xstar)
        err_res[i] = abs(f(x[i]))
        diff_x[i-1] = abs(x[i-1]-x[i])
        print(f"it = {i}, xn = {x[i]:14.8e}, en = {err[i]:14.8e}, |f(xn)| = {err_res[i]:14.8e}, |xn - xn-1| = {diff_x[i-1]:14.8e}" )
        ##if ( f(x[i]) < tol ): break
        if ( err[i] < tol ): break

    return x[0:i+1], err[0:i+1], err_res[0:i+1], diff_x[0:i]

In [ ]:
xsol, err, err_res, diff_x = newton(f, df, x0=4., xstar=2.0945514815423265)

In [ ]:
xmin = 1.9; xmax = 4.1
x = np.linspace(1.9, 4.1, 100)

fig = make_subplots(rows=2, cols=1)

fig.add_shape(type="line", x0=xmin, y0=0, x1=xmax, y1=0, line=dict(color="black",width=2), row=1, col=1)
fig.add_shape(type="line", x0=xmin, y0=0, x1=xmax, y1=0, line=dict(color="black",width=2), row=2, col=1)

fig.add_trace(go.Scatter(x=x, y=f(x), name='f(x)', legendgroup = '1', showlegend=True), row=1, col=1)
g = lambda x : x - f(x)/df(x)
fig.add_trace(go.Scatter(x=x, y=g(x), name='g(x) = x - f(x)/df(x)', line_color='rgb(76,114,176)', legendgroup = '2'), row=2, col=1)
fig.add_trace(go.Scatter(x=x, y=x, line_color="black", name='y=x', legendgroup = '2'), row=2, col=1)

for i in range(xsol.size-1):
    fig.add_trace(go.Scatter(x=[xsol[i], xsol[i]], y=[0, f(xsol[i])], showlegend=False, visible=False,
                             line=dict(color='rgb(221,132,82)', width=2, dash='dash')), row=1, col=1)
    fig.add_trace(go.Scatter(x=[xsol[i], xsol[i+1]], y=[f(xsol[i]), 0], showlegend=False, visible=False,
                             line=dict(color='rgb(221,132,82)', width=2, dash='dash')), row=1, col=1)
    fig.add_trace(go.Scatter(x=[xsol[i], xsol[i]], y=[0, g(xsol[i])], showlegend=False, visible=False,
                              line=dict(color='rgb(221,132,82)', width=2, dash='dash')), row=2, col=1)
    fig.add_trace(go.Scatter(x=[xsol[i], xsol[i+1]], y=[g(xsol[i]), xsol[i+1]], showlegend=False, visible=False,
                             line=dict(color='rgb(221,132,82)', width=2, dash='dash')), row=2, col=1)

# Create and add slider
steps = []
for i in range(xsol.size-1):
    args = [{"visible": [(el<4*i+3) for el in range(len(fig.data))]}]
    step = dict(method="update", label = f"{i}", args=args)
    steps.append(step)
sliders = [dict(currentvalue={'prefix': 'Iteration nb = '}, steps=steps)]

fig.update_xaxes(range=[1.9, 4.1], tickmode = 'array', tickvals=xsol[:4], tickformat='.2f')    
fig.update_yaxes(range=[-0.1, 4.1], row=1, col=2)
legend = dict(x=0.01, bgcolor="rgba(0,0,0,0)")
fig.update_layout(legend=legend, sliders=sliders, title="Newton iterations", height=800, legend_tracegroupgap=320)    
fig.show()

## Convergence of Newton's method

In [ ]:
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=np.arange(err.size), y=err, name='error', mode='markers')) 
fig1.add_trace(go.Scatter(x=np.arange(err_res.size), y=err_res, name='residual error', mode='markers')) 
fig1.add_trace(go.Scatter(x=np.arange(1, err_res.size), y=diff_x[:], name='difference of the iterates', mode='markers')) 
fig1.update_xaxes(title='Iteration')    
fig1.update_yaxes(type="log", exponentformat='e', title="Error")
fig1.update_layout(title="Convergence history", legend=dict(orientation="h", y=1.1))
fig1.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=err[:-1:], y=err[1:], name='error', mode='markers')) 
fig2.add_trace(go.Scatter(x=err[:-1], y=err[:-1], name='line of slope -1', mode='lines', line_dash='dot')) 
fig2.add_trace(go.Scatter(x=err[:-1], y=err[:-1]*err[:-1], name='line of slope -2', mode='lines', line_dash='dot')) 
fig2.update_xaxes(type="log", exponentformat='e', title="Error at iteration k")    
fig2.update_yaxes(type="log", exponentformat='e', title="Error at iteration k+1") 
fig2.update_layout(title="Order of convergence", legend=dict(orientation="h", y=1.1))
fig2.show()

# Secant method

We propose to test the secant method on the same function:

$$f(x) = x^3 -2 x -5 $$

In [ ]:
def secant(f, x0, x1, xstar, tol=1.e-12, nitmax=50):
   
    # initialisation
    x = np.zeros(nitmax+2)
    err = np.zeros(nitmax+2)
    err_res = np.zeros(nitmax+2)
    diff_x = np.zeros(nitmax+1)
    x[0] = x0
    x[1] = x1
    err[0] = abs(x0 - xstar)
    err[1] = abs(x1 - xstar)
    err_res[0] = abs(f(x0))
    err_res[1] = abs(f(x1))
    diff_x[0] = abs(x1-x0)
    
    # iteration of the secant method        
    for i in range(2, nitmax+2):
        x[i] = x[i-1] - ((x[i-1] - x[i-2])/(f(x[i-1]) - f(x[i-2])))*f(x[i-1])
        err[i] = abs(x[i]-xstar)
        err_res[i] = abs(f(x[i]))
        diff_x[i-1] = abs(x[i-1]-x[i])
        print(f"it = {i-1}, xn = {x[i]:16.8e}, en = {err[i]:16.8e}, |f(xn)| = {err_res[i]:16.8e}, |xn - xn-1| = {diff_x[i-1]:16.8e}" )
        #print(i, x[i], err_res[i], err[i])
        ##if ( f(x[i]) < tol ): break
        if ( err[i] < tol ): break

    return x[0:i+1], err[0:i+1], err_res[0:i+1], diff_x[0:i]

## Iteration of the secant method

In [ ]:
xsol, err, err_res, diff_x = secant(f, x0=4, x1=3.8, xstar=2.0945514815423265)

In [ ]:
xmin = 1.9; xmax = 4.1
x = np.linspace(1.9, 4.1, 100)

fig = go.Figure()

fig.add_shape(type="line", x0=xmin, y0=0, x1=xmax, y1=0, line=dict(color="black",width=2))
fig.add_trace(go.Scatter(x=x, y=f(x), name='f(x)', showlegend=True))

for i in range(xsol.size-2):
    fig.add_trace(go.Scatter(x=[xsol[i], xsol[i+1], xsol[i+2]], y=[f(xsol[i]), f(xsol[i+1]), 0], showlegend=False, visible=False,
                             line=dict(color='rgb(221,132,82)', width=2, dash='dash')))

# Create and add slider
steps = []
for i in range(xsol.size-1):
    args = [{"visible": [(el<i+1) for el in range(len(fig.data))]}]
    step = dict(method="update", label = f"{i}", args=args)
    steps.append(step)
sliders = [dict(currentvalue={'prefix': 'Iteration nb = '}, steps=steps)]    

legend = dict(x=0.01, bgcolor="rgba(0,0,0,0)")    
fig.update_layout(legend=legend, sliders=sliders, title="Secant method iterations", height=500)    
fig.show()                             

## Convergence of the secant method

In [ ]:
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=np.arange(err.size), y=err, name='error', mode='markers')) 
fig1.add_trace(go.Scatter(x=np.arange(err_res.size), y=err_res, name='residual error', mode='markers')) 
fig1.add_trace(go.Scatter(x=np.arange(1, err.size), y=diff_x[:], name='difference of the iterates', mode='markers')) 
fig1.update_xaxes(title='Iteration')    
fig1.update_yaxes(type="log", exponentformat='e', title="Error")    
fig1.update_layout(title="Convergence history", legend=dict(orientation="h", y=1.1))    
fig1.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=err[:-1:], y=err[1:], name='error', mode='markers')) 
fig2.add_trace(go.Scatter(x=err[:-1], y=err[:-1], name='line of slope -1', mode='lines', line_dash='dot')) 
fig2.add_trace(go.Scatter(x=err[:-1], y=err[:-1]*err[:-1], name='line of slope -2', mode='lines', line_dash='dot')) 
fig2.update_xaxes(type="log", exponentformat='e', title="Error at iteration k")    
fig2.update_yaxes(type="log", exponentformat='e', title="Error at iteration k+1") 
fig2.update_layout(title="Order of convergence", legend=dict(orientation="h", y=1.1))    
fig2.show()